In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [3]:
df = pd.read_csv("../data/processed/campaign_data_feature_engineered.csv")
df.head()

,campaign_type,platform,region,device,age_group,season,campaign_date,budget,impressions,clicks,...,cpa,revenue_per_click,revenue_per_conversion,profit_margin,engagement_per_session,customer_rating_pct,month,quarter,day_of_week,is_weekend
0,search,facebook,north,desktop,55+,summer,2023-07-22,38768.0,362738,12261,...,53.490546,275.512810,4982.393156,98.926409,32.147260,100.0,7,3,Saturday,1
1,affiliate,facebook,east,mobile,35-44,festive,2025-10-22,20507.0,200613,8900,...,22.252956,1085.222791,11749.979124,99.810613,85.966443,70.0,10,4,Wednesday,0
2,search,google ads,west,mobile,55+,summer,2023-01-11,30433.0,309070,6272,...,90.072141,226.876433,4008.363352,97.752895,100.941667,78.0,1,1,Wednesday,0
3,email,youtube,west,desktop,55+,festive,2024-06-15,46373.0,1188007,38502,...,32.480692,306.084719,9828.918966,99.669540,454.861272,66.0,6,2,Saturday,1
4,influencer,x,west,desktop,55+,winter,2025-10-06,65564.0,335654,9754,...,670.368125,59.004739,5995.127292,88.818117,237.175439,100.0,10,4,Monday,0


In [4]:
drop_cols = [
    'revenue',
    'profit',
    'revenue_per_click',
    'revenue_per_conversion',
    'profit_margin'
]

df = df.drop(columns=drop_cols)

In [5]:
X = df.drop('roi', axis=1)
y = df['roi']

In [6]:
X.select_dtypes(include='object').columns

Index(['campaign_type', 'platform', 'region', 'device', 'age_group', 'season',
       'campaign_date', 'day_of_week'],
      dtype='object')

In [7]:
X = pd.get_dummies(X,drop_first=True)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [9]:
print(X_train.shape)
print(X_test.shape)

(40000, 1147)
(10000, 1147)


# Linear Regression

In [10]:
# Create model
lr = LinearRegression()

In [11]:
# Train model
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [12]:
# Predictions
y_pred_lr = lr.predict(X_test)

In [30]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

In [31]:
print("Linear Regression Performance")
print(f"MAE  : {mae_lr:.4f}")
print(f"RMSE : {rmse_lr:.4f}")
print(f"R²   : {r2_lr:.4f}")

Linear Regression Performance
MAE  : 8186.0661
RMSE : 16362.6945
R²   : 0.4232


# Random Forest

In [15]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

mae_rf_all = mean_absolute_error(y_test, y_pred_rf)
rmse_rf_all = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf_all = r2_score(y_test, y_pred_rf)

In [16]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

In [17]:
top_20_features = feature_importance.head(20)
selected_features = top_20_features.index.tolist()
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

print(f"Original X_train shape: {X_train.shape}")
print(f"Selected X_train shape: {X_train_selected.shape}")
print(f"Original X_test shape: {X_test.shape}")
print(f"Selected X_test shape: {X_test_selected.shape}")

Original X_train shape: (40000, 1147)
Selected X_train shape: (40000, 20)
Original X_test shape: (10000, 1147)
Selected X_test shape: (10000, 20)


In [18]:
rf_selected = RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=-1)

In [19]:
rf_selected.fit(X_train_selected, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [20]:
y_pred_selected = rf_selected.predict(X_test_selected)

In [21]:
# ==========================
# Random Forest Evaluation
# ==========================

mae_rf = mean_absolute_error(y_test, y_pred_selected)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_selected))
r2_rf = r2_score(y_test, y_pred_selected)

print("Random Forest Performance")
print(f"MAE  : {mae_rf:.4f}")
print(f"RMSE : {rmse_rf:.4f}")
print(f"R²   : {r2_rf:.4f}")

Random Forest Performance
MAE  : 6117.0236
RMSE : 14434.0013
R²   : 0.5512


In [22]:
print("Original Features :", X_train.shape[1])
print("Selected Features :", X_train_selected.shape[1])
print("Selected Features List:")
print(selected_features)

Original Features : 1147
Selected Features : 20
Selected Features List:
['cpa', 'ctr', 'cpc', 'bounce_rate', 'cpl', 'session_duration_sec', 'campaign_date_2024-01-11', 'season_winter', 'engagement_rate', 'conversion_rate', 'lead_conversion_rate', 'shares', 'season_summer', 'comments', 'engagement_per_session', 'season_monsoon', 'campaign_date_2023-10-16', 'impressions', 'leads', 'likes']


In [23]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest (All Features)",
        "Random Forest (Top 20 Features)"
    ],
    "MAE": [
        mae,
        mae_rf_all,
        mae_rf
    ],
    "RMSE": [
        rmse,
        rmse_rf_all,
        rmse_rf
    ],
    "R² Score": [
        r2,
        r2_rf_all,
        r2_rf
    ]
})

comparison_df

,Model,MAE,RMSE,R² Score
0,Linear Regression,8186.066088,16362.694503,0.423191
1,Random Forest (All Features),6063.720405,14474.880927,0.548610
2,Random Forest (Top 20 Features),6117.023600,14434.001300,0.551156


In [24]:
feature_importance.head(20)

cpa                         0.597943
ctr                         0.019099
cpc                         0.016299
bounce_rate                 0.015560
cpl                         0.013870
session_duration_sec        0.013824
campaign_date_2024-01-11    0.013726
season_winter               0.013306
engagement_rate             0.012802
conversion_rate             0.011399
lead_conversion_rate        0.011206
shares                      0.011199
season_summer               0.010976
comments                    0.010859
engagement_per_session      0.010534
season_monsoon              0.008559
campaign_date_2023-10-16    0.008386
impressions                 0.008377
leads                       0.008249
likes                       0.008054
dtype: float64

In [25]:
from xgboost import XGBRegressor

xgb_tuned = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42
)

xgb_tuned.fit(X_train_selected, y_train)

print("Train R²:", xgb_tuned.score(X_train_selected, y_train))
print("Test R² :", xgb_tuned.score(X_test_selected, y_test))

Train R²: 0.6618916681527993
Test R² : 0.5817338435834366


In [26]:
# ======================================
# LEAKAGE EXPERIMENT
# ======================================

from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# Reload original feature engineered dataset
df_leak = pd.read_csv(
    "../data/processed/campaign_data_feature_engineered.csv"
)

# Create X and y
X_leak = df_leak.drop('roi', axis=1)
y_leak = df_leak['roi']

# One Hot Encoding
X_leak = pd.get_dummies(
    X_leak,
    drop_first=True
)

# Train Test Split
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak,
    y_leak,
    test_size=0.20,
    random_state=42
)

# XGBoost Model
xgb_leak = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42
)

xgb_leak.fit(
    X_train_leak,
    y_train_leak
)

# Predictions
y_pred_leak = xgb_leak.predict(X_test_leak)

# Results
print("=" * 50)
print("WITH LEAKAGE FEATURES")
print("=" * 50)

print(
    "Train R²:",
    xgb_leak.score(
        X_train_leak,
        y_train_leak
    )
)

print(
    "Test R²:",
    r2_score(
        y_test_leak,
        y_pred_leak
    )
)

WITH LEAKAGE FEATURES
Train R²: 0.9965608263112519
Test R²: 0.9671921066061646


In [27]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    xgb_tuned,
    X,
    y,
    cv=5,
    scoring='r2'
)

print("CV R² Scores:", scores)
print("Mean CV R²:", scores.mean())

CV R² Scores: [0.63109637 0.60388671 0.58397278 0.57550666 0.59744695]
Mean CV R²: 0.5983818975697929


In [36]:
import pandas as pd
import numpy as np

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest (All Features)",
        "Random Forest (Top 20 Features)",
        "XGBoost"
    ],

    "MAE": [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_selected),
        mean_absolute_error(y_test, y_pred_leak)
    ],

    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_selected)),
        np.sqrt(mean_squared_error(y_test, y_pred_leak))
    ],

    "R2": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_selected),
        r2_score(y_test, y_pred_leak)
    ]
})

results

,Model,MAE,RMSE,R2
0,Linear Regression,8186.066088,16362.694503,0.423191
1,Random Forest (All Features),6063.720405,14474.880927,0.548610
2,Random Forest (Top 20 Features),6117.023600,14434.001300,0.551156
3,XGBoost,530.975835,3902.367071,0.967192


In [37]:
results.to_csv(
    "../model/model_comparison.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.


In [38]:
feature_importance.to_csv(
    "../model/feature_importance.csv",
    index=False
)

print("Feature Importance saved successfully.")

Feature Importance saved successfully.


In [39]:
top_20_features.to_csv(
    "../model/top_20_features.csv",
    index=False
)

print("Top 20 Features saved successfully.")

Top 20 Features saved successfully.


In [40]:
import joblib

joblib.dump(
    X_train.columns.tolist(),
    "../model/model_features.pkl"
)

['../model/model_features.pkl']